# 视频分类 SlowFast

In [ ]:
import os
import cv2
import decord
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from pytorchvideo.models.hub import slowfast_r50

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## 一、处理数据

In [ ]:
ucf_root_dir = r"/mnt/Data/daihongliang"
ucf_dataset_name = "UCF-101"
ucf_processed_name = "UCF-101-Processed"
ucf_train_test_file_dir = "ucfTrainTestlist"

In [ ]:
def get_train_test_lists(ucf_root_dir, ucf_dataset_name):
    data_dir = os.path.join(ucf_root_dir, ucf_dataset_name)
    class_list = os.listdir(data_dir)
    data_list = []

    for i in range(len(class_list)):
        class_item_dir = os.path.join(data_dir, class_list[i])
        data_list.extend([os.path.join(class_item_dir, it) for it in os.listdir(class_item_dir)])

    train_list, test_list = train_test_split(data_list, test_size=0.2)

    return train_list, test_list

In [ ]:
train_data, test_data = get_train_test_lists(ucf_root_dir, ucf_dataset_name)

In [ ]:
train_data[:5]

In [ ]:
test_data[:5]

## 二、编写 Torch 的 Dataset

In [ ]:
class UCFDataset(Dataset):
    def __init__(self, ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=False):
        self.num_frames = 32

        # 读取训练标签
        self.classes = {}
        classes_file = os.path.join(ucf_root_dir, ucf_train_test_file_dir, 'classInd.txt')
        with open(classes_file) as fin:
            for row in fin:
                line_info = row.strip().split(' ')
                self.classes[line_info[1]] = int(line_info[0]) - 1

        # 读取数据
        train_data, test_data = get_train_test_lists(ucf_root_dir, ucf_dataset_name)
        self.data = train_data if train else test_data
        
        # 数据处理
        self.transform = T.Compose([
            # T.ToTensor(),
            T.Resize(256),
            lambda x: x / 255.0,
            T.CenterCrop(224),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        vr = decord.VideoReader(self.data[i], num_threads=4)
        total_frames = len(vr)

        if total_frames <= self.num_frames:
            frames = vr.get_batch(range(len(vr))).asnumpy()
        else:
            max_start = total_frames - self.num_frames
            start_frame = random.randint(0, max_start)
            
            frame_indices = list(range(start_frame, start_frame + self.num_frames))
            frames = vr.get_batch(frame_indices).asnumpy()
            

        frames = torch.tensor(frames).permute(0, 3, 1, 2).float()
        frames = self.transform(frames)

        return frames, self.classes[self.data[i].split('/')[-2]]

In [ ]:
trainset = UCFDataset(ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=True)
testset = UCFDataset(ucf_root_dir, ucf_dataset_name, ucf_train_test_file_dir, train=False)

In [ ]:
d, l = trainset.__getitem__(123)
print(d.shape, l)

## 三、编写 Torch 的 DataLoader

In [ ]:
def collate_fn(batch):
    data = []
    label = []
    
    # 筛选总帧数小于64的样本
    for d, l in batch:
        num_frames = d.shape[0]
        if num_frames < 32:
            continue  # 跳过帧数不足的样本
        
        data.append(d)
        label.append(l)
    
    # 处理批量中无有效样本的情况
    if len(data) == 0:
        return torch.tensor([]), torch.tensor([])
    
    slow_tensor = [item.clone()[::4] for item in data]
    fast_tensor = [item.clone() for item in data]
    
    # 组装张量（所有样本采样后帧数相同）
    slow_tensor = torch.stack(slow_tensor, dim=0).type(torch.FloatTensor).transpose(1, 2).contiguous()
    fast_tensor = torch.stack(fast_tensor, dim=0).type(torch.FloatTensor).transpose(1, 2).contiguous()
    label_tensor = torch.tensor(label).type(torch.FloatTensor)
    
    return [slow_tensor, fast_tensor], label_tensor

In [ ]:
trainloader = DataLoader(trainset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=4)
testloader = DataLoader(testset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=4)

In [ ]:
batch_data, batch_labels = next(iter(trainloader))
print("数据形状:", batch_data[0].shape, batch_data[1].shape)
print("标签形状:", batch_labels.shape)

## 四、设计网络

In [ ]:
class SlowFast(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        # 加载预训练的SlowFast R50模型
        self.model = slowfast_r50(pretrained=pretrained)
        
        # 替换分类头以适应你的类别数
        in_features = self.model.blocks[6].proj.in_features
        self.model.blocks[6].proj = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.model(x)

In [ ]:
net = SlowFast(101)
o = net(batch_data)
print(o.shape)

## 五、构建训练代码

In [ ]:
seed = 1
torch.manual_seed(seed)  # 为CPU设置随机种子
torch.cuda.manual_seed(seed)  # 为当前GPU设置随机种子
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU，为所有GPU设置随机种子
np.random.seed(seed)  # Numpy module.
random.seed(seed)  # Python random module.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [ ]:
save_root = "./checkpoints/SlowFast"
num_classes = 101
learning_rate = 1e-4
train_epochs = 100

In [ ]:
# 创建检查目录
os.makedirs(os.path.join(save_root, 'acc'), exist_ok=True)
os.makedirs(os.path.join(save_root, 'loss'), exist_ok=True)

In [ ]:
model = SlowFast(num_classes)
if torch.cuda.device_count() > 1:
    print(f"使用 {torch.cuda.device_count()} 个GPU进行训练。")
    model = nn.DataParallel(model)  # 包装模型
model = model.cuda()

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)
criterion = nn.CrossEntropyLoss()

In [ ]:
def calculate_metrics(y_true, y_pred):
    """计算各种评估指标"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    return accuracy, precision, recall, micro_f1, macro_f1

In [ ]:
best_accuracy = 0
min_loss = float('inf')

for epoch in range(1, train_epochs + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{train_epochs}")
    print(f"{'='*60}")
    
    # 更新学习率
    # scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"当前学习率: {current_lr}")

    # 训练循环
    model.train()
    train_loss, train_correct, train_num = 0.0, 0, 0
    train_pred, train_label = [] , []
    for _, (x, y) in enumerate(tqdm(trainloader, desc="训练中")):
        if not y.shape[0]:
            continue
        x, y = [x[0].cuda(), x[1].cuda()], y.long().cuda()
        
        # 梯度清零
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(x)
        loss = criterion(outputs, y)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        # 统计信息
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_correct += (predicted == y).sum().item()
        train_num += y.size(0)
        
        train_pred.extend(predicted.cpu().numpy())
        train_label.extend(y.cpu().numpy())

    train_avg_loss = train_loss / len(trainloader)
    train_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(train_label, train_pred)
    print(f"Loss: {train_avg_loss:.4f}, Accuracy: {train_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

    # 测试循环
    model.eval()
    test_loss, test_correct_num, test_num = 0.0, 0, 0
    test_pred, test_label = [], []
    
    with torch.no_grad():
        for _, (x, y) in enumerate(tqdm(testloader, desc="测试中")):
            if not y.shape[0]:
                continue
            x, y = [x[0].cuda(), x[1].cuda()], y.long().cuda()
            
            outputs = model(x)
            loss = criterion(outputs, y)
            
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_correct_num += (predicted == y).sum().item()
            test_num += y.size(0)
            
            test_pred.extend(predicted.cpu().numpy())
            test_label.extend(y.cpu().numpy())
    
    test_avg_loss = test_loss / len(testloader)
    test_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(test_label, test_pred)
    print(f"Loss: {test_avg_loss:.4f}, Accuracy: {test_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

     # 保存最佳准确率模型
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        save_path = os.path.join(save_root, 'acc', f'best_acc.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
    
    # 保存最低损失模型
    if test_avg_loss < min_loss:
        min_loss = test_avg_loss
        save_path = os.path.join(save_root, 'loss', f'best_loss.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")

    
    # 正确率超过99%时结束训练
    if test_accuracy >= 0.99:
        print(f"训练准确率已达到 {test_accuracy:.2%}，提前终止训练")
        save_path = os.path.join(save_root, f'early_stop_epoch_{epoch}.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
        break
